# 02 — Object Detection with YOLOv8

YOLO (You Only Look Once) runs a single neural network pass over the image and outputs bounding boxes + class labels for every detected object — hence "you only look once".

This notebook covers:
- Loading a fine-tuned YOLOv8 model
- Running inference on a single frame
- Understanding the output format
- Visualising detections

**Connection to the pipeline:** `tracker.detect_frames()` in `football_analysis/trackers/tracker.py`

In [ ]:
import sys
sys.path.append('..')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from ultralytics import YOLO

from football_analysis.utils.video_utils import read_video

In [ ]:
MODEL_PATH = '../models/best.pt'
VIDEO_PATH = '../input_videos/08fd33_4.mp4'

model = YOLO(MODEL_PATH)
print('Classes:', model.names)

## Running inference on a single frame

`model.predict()` accepts a list of frames (or file paths).  
`conf=0.1` is a low confidence threshold — we keep borderline detections and let the tracker smooth them out.

In [ ]:
import cv2
cap = cv2.VideoCapture(VIDEO_PATH)
ret, frame = cap.read()
cap.release()

results = model.predict([frame], conf=0.1)
result = results[0]   # one result per image in the list

print(type(result))
print(dir(result.boxes))

## Unpacking the result

Each detected object has:
- `xyxy` — bounding box corners `[x1, y1, x2, y2]` in pixel coords
- `conf` — confidence score 0-1
- `cls` — integer class index

In [ ]:
boxes = result.boxes

print(f'Detected {len(boxes)} objects')
print()

for i, box in enumerate(boxes):
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    conf  = box.conf[0].item()
    cls   = int(box.cls[0].item())
    label = result.names[cls]
    print(f'  [{i:2d}] {label:12s}  conf={conf:.2f}  box=({x1:.0f},{y1:.0f},{x2:.0f},{y2:.0f})')

## Visualising detections

In [ ]:
CLASS_COLOURS = {
    'player':     'cyan',
    'goalkeeper': 'yellow',
    'referee':    'orange',
    'ball':       'red',
}

fig, ax = plt.subplots(figsize=(16, 9))
ax.imshow(frame[:, :, ::-1])

for box in boxes:
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    cls   = int(box.cls[0].item())
    label = result.names[cls]
    colour = CLASS_COLOURS.get(label, 'white')

    rect = patches.Rectangle(
        (x1, y1), x2 - x1, y2 - y1,
        linewidth=1.5, edgecolor=colour, facecolor='none'
    )
    ax.add_patch(rect)
    ax.text(x1, y1 - 3, label, color=colour, fontsize=7, fontweight='bold')

ax.set_title('YOLOv8 detections — frame 0')
ax.axis('off')
plt.tight_layout()
plt.show()

## Why do we batch frames?

Running inference one frame at a time wastes GPU setup time.  
The pipeline batches 20 frames at once — same number of model calls, much higher throughput.

In [ ]:
import time
import cv2

cap = cv2.VideoCapture(VIDEO_PATH)
test_frames = []
for _ in range(20):
    ret, f = cap.read()
    if ret:
        test_frames.append(f)
cap.release()

# One at a time
t0 = time.time()
for f in test_frames:
    model.predict([f], conf=0.1, verbose=False)
t_single = time.time() - t0

# All at once
t0 = time.time()
model.predict(test_frames, conf=0.1, verbose=False)
t_batch = time.time() - t0

print(f'One-at-a-time: {t_single:.2f}s')
print(f'Batch of 20  : {t_batch:.2f}s')
print(f'Speedup      : {t_single / t_batch:.1f}x')

## Why `conf=0.1`?

Lower confidence → more detections, including uncertain ones.  
ByteTrack (next notebook) handles false positives by requiring consistency across frames.  
A missed detection is worse than a false detection in a tracking context.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
thresholds = [0.1, 0.3, 0.5]

for ax, thresh in zip(axes, thresholds):
    res = model.predict([frame], conf=thresh, verbose=False)[0]
    ax.imshow(frame[:, :, ::-1])
    for box in res.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        cls = int(box.cls[0].item())
        colour = CLASS_COLOURS.get(res.names[cls], 'white')
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                  linewidth=1.5, edgecolor=colour, facecolor='none')
        ax.add_patch(rect)
    ax.set_title(f'conf={thresh}  →  {len(res.boxes)} detections')
    ax.axis('off')

plt.suptitle('Effect of confidence threshold', fontsize=13)
plt.tight_layout()
plt.show()

## Key takeaways

| Concept | Detail |
|---|---|
| YOLO output | `xyxy` bounding boxes + class index + confidence per detection |
| Batch inference | ~2-5× faster than one-at-a-time for the same number of frames |
| `conf=0.1` | Deliberately low — ByteTrack filters noise across time |
| Goalkeeper hack | We remap goalkeeper → player class before tracking so ByteTrack treats them as one class |

**Next:** `03_tracking.ipynb` — giving each player a stable ID across frames